# Lab 03: From Frozen Features to Fine-Tuning

Using a pretrained ResNet18 on the EuroSAT satellite dataset and comparing two runs:

1. Feature extraction: freeze the whole backbone and only train a new last layer.
2. Fine-tuning: also unfreeze `layer4` and train it with a smaller learning rate.

Both runs use the same split, seed, augmentation and number of epochs.

In [1]:
import os
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.datasets.utils import download_and_extract_archive

SEED = 0
EPOCHS = 5
BATCH_SIZE = 32
TRAIN_PER_CLASS = 200
VAL_PER_CLASS = 50
HEAD_LR = 1e-3
BACKBONE_LR = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
os.makedirs("figures", exist_ok=True)

device: cpu


## 1. Dataset and transforms

The transforms are copied from the starter file.

In [2]:
# from lab03_starter.py
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

EuroSAT has 27,000 images in 10 classes. The full set is too slow to train on my laptop, so I take 200 train and 50 val images per class.

I load the folder twice, once with each transform, so the training images only get `train_transform` and the validation images only get `eval_transform`. The torchvision download link for EuroSAT is broken, so I download it from Hugging Face instead.

In [3]:
DATA_DIR = "data/2750"
EUROSAT_URL = ("https://huggingface.co/datasets/torchgeo/eurosat/resolve/"
               "c877bcd43f099cd0196738f714544e355477f3fd/EuroSAT.zip")

if not os.path.isdir(DATA_DIR):
    download_and_extract_archive(EUROSAT_URL, "data")

train_data = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_data = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
classes = train_data.classes
num_classes = len(classes)

# pick the same train/val images for both runs
g = torch.Generator().manual_seed(SEED)
targets = torch.tensor(train_data.targets)
train_idx, val_idx = [], []
for c in range(num_classes):
    idx = (targets == c).nonzero().squeeze(1)
    idx = idx[torch.randperm(len(idx), generator=g)]
    train_idx += idx[:TRAIN_PER_CLASS].tolist()
    val_idx += idx[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS].tolist()

train_set = Subset(train_data, train_idx)
val_set = Subset(val_data, val_idx)

print("classes:", classes)
print(f"training images: {len(train_set)}, validation images: {len(val_set)}")
print("overlap between train and val:", len(set(train_idx) & set(val_idx)))

classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
training images: 2000, validation images: 500
overlap between train and val: 0
